# AI Business Copilot (Multi-Agent System)

An autonomous AI system that analyzes business data using a planner → worker → evaluator loop.

## Features

- Data analysis & insights
- Risk detection
- Growth recommendations

In [41]:
# !pip install langchain langgraph langchain-openai gradio python-dotenv

In [42]:
import os
import uuid
from typing import Annotated, Optional, Any
from typing_extensions import TypedDict
from pydantic import BaseModel

import gradio as gr
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

from langchain.agents import Tool
from langchain_experimental.tools import PythonREPLTool
from langchain_community.utilities import GoogleSerperAPIWrapper

load_dotenv()

MODEL = "openai/gpt-4o-mini"
BASE_URL = "https://openrouter.ai/api/v1"
API_KEY = os.getenv("OPENAI_API_KEY")

serper = GoogleSerperAPIWrapper()

In [43]:
serper = GoogleSerperAPIWrapper()

def get_tools():
    return [
        Tool(
            name="search",
            func=serper.run,
            description="Search market trends or benchmarks"
        ),
        PythonREPLTool()
    ]

In [44]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    success_criteria: str
    success: bool
    final_response: Optional[str]

In [45]:
class EvalOutput(BaseModel):
    feedback: str
    success: bool

In [46]:
class BusinessCopilot:
    def __init__(self):
        self.id = str(uuid.uuid4())

    async def setup(self):
        # Memory
        self.memory_ctx = AsyncSqliteSaver.from_conn_string("business_memory.db")
        self.memory = await self.memory_ctx.__aenter__()

        # Tools & LLMs
        self.tools = get_tools()
        self.llm = ChatOpenAI(model=MODEL, base_url=BASE_URL, api_key=API_KEY)
        self.llm_with_tools = self.llm.bind_tools(self.tools)
        self.eval_llm = self.llm.with_structured_output(EvalOutput)

        # Build graph
        await self.build_graph()

    # Worker (Business Analyst)
    def worker(self, state: State):
        system = f"""
You are an AI Business Analyst.
Analyze business data for:
- Insights
- Risks
- Growth strategies
Success criteria: {state['success_criteria']}
"""
        messages = [SystemMessage(content=system)] + state["messages"]
        res = self.llm_with_tools.invoke(messages)
        return {"messages": [res], "final_response": res.content}

    # Worker router
    def worker_router(self, state: State):
        last = state["messages"][-1]
        if hasattr(last, "tool_calls") and last.tool_calls:
            return "tools"
        return "evaluator"

    # Evaluator
    def evaluator(self, state: State):
        prompt = f"""
Evaluate this response:

{state['final_response']}

Success criteria:
{state['success_criteria']}
"""
        res = self.eval_llm.invoke([HumanMessage(content=prompt)])
        return {"success": res.success, "messages": [AIMessage(content=res.feedback)]}

    # Evaluation router
    def route_eval(self, state: State):
        if state["success"]:
            return "END"
        return "worker"

    # Build the state graph
    async def build_graph(self):
        graph = StateGraph(State)
        graph.add_node("worker", self.worker)
        graph.add_node("tools", ToolNode(self.tools))
        graph.add_node("evaluator", self.evaluator)

        graph.add_edge(START, "worker")
        graph.add_conditional_edges(
            "worker",
            self.worker_router,
            {"tools": "tools", "evaluator": "evaluator"}
        )
        graph.add_edge("tools", "worker")
        graph.add_conditional_edges(
            "evaluator",
            self.route_eval,
            {"worker": "worker", "END": END}
        )
        self.graph = graph.compile(checkpointer=self.memory)

    # Run a query
    async def run(self, message, criteria):
        state = {
            "messages": [HumanMessage(content=message)],
            "success_criteria": criteria,
            "success": False,
            "final_response": None
        }
        result = await self.graph.ainvoke(state, config={"configurable": {"thread_id": self.id}})
        return result.get("final_response", "")

In [47]:
def planner(self, state: State):
    prompt = f"""
User request:
{state['messages'][-1].content}

Success criteria:
{state['success_criteria']}

Break into steps for business analysis.
"""

    res = self.planner_llm.invoke([HumanMessage(content=prompt)])

    return {
        "messages": [AIMessage(content=str(res.steps))],
        "plan": res
    }

In [48]:
def worker(self, state: State):
        system = f"""
You are a senior AI Business Analyst.

Analyze the business data and produce:
- Key insights
- Risks
- Growth opportunities
- Executive summary

Success criteria:
{state['success_criteria']}
"""

        messages = [SystemMessage(content=system)] + state["messages"]

        response = self.llm_with_tools.invoke(messages)

        return {
            "messages": [response],
            "final_response": response.content
        }

In [49]:
def worker_router(self, state: State):
        last = state["messages"][-1]

        if hasattr(last, "tool_calls") and last.tool_calls:
            return "tools"

        return "evaluator"

In [50]:
def evaluator(self, state: State):
        prompt = f"""
Evaluate this response:

{state['final_response']}

Criteria:
{state['success_criteria']}
"""

        result = self.eval_llm.invoke([HumanMessage(content=prompt)])

        return {
            "success": result.success,
            "messages": [AIMessage(content=result.feedback)]
        }

In [51]:
def route_eval(self, state: State):
        if state["success"]:
            return "END"
        return "worker"

In [52]:
async def build_graph(self):
        graph = StateGraph(State)

        graph.add_node("worker", self.worker)
        graph.add_node("tools", ToolNode(self.tools))
        graph.add_node("evaluator", self.evaluator)

        graph.add_edge(START, "worker")

        graph.add_conditional_edges(
            "worker",
            self.worker_router,
            {
                "tools": "tools",
                "evaluator": "evaluator"
            }
        )

        graph.add_edge("tools", "worker")

        graph.add_conditional_edges(
            "evaluator",
            self.route_eval,
            {
                "worker": "worker",
                "END": END
            }
        )

        self.graph = graph.compile(checkpointer=self.memory)

In [53]:
async def run(self, message, criteria):
        state = {
            "messages": [HumanMessage(content=message)],
            "success_criteria": criteria,
            "success": False,
            "final_response": None
        }

        result = await self.graph.ainvoke(
            state,
            config={"configurable": {"thread_id": self.id}}
        )

        return result.get("final_response", "")

In [54]:
async def setup():
    bot = BusinessCopilot()
    await bot.setup()
    return bot


async def run_ui(bot, message, criteria, history):
    if not bot:
        return history, None

    res = await bot.run(message, criteria)

    return history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": res}
    ], bot


with gr.Blocks() as demo:
    gr.Markdown("# 🧠 AI Business Copilot")

    bot = gr.State()
    chat = gr.Chatbot(type="messages")

    msg = gr.Textbox(label="Business Idea / Data")
    criteria = gr.Textbox(label="Success Criteria")

    btn = gr.Button("Analyze")

    demo.load(setup, [], bot)

    btn.click(run_ui, [bot, msg, criteria, chat], [chat, bot])

demo.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


## Future Improvements
- Real data connectors (Stripe, DB)
- Streaming responses
- Dashboard visualization (Next.js)